# Tests des fonctions de calcul de la saturation

In [27]:
import json
from datetime import date, timedelta

import pandas as pd

#from saturation_image_quali_prod import (
from e2_e3_main03_09 import (
    filter_statuses_sessions,
    get_sampled_state_poc,
    get_sampled_state_poc_for_chunk,
)
from pandas import NamedAgg
from utils_main03_09 import (
    to_sampled_sessions,
    to_sampled_state_grp,
    to_sampled_state_poc,
    to_sampled_statuses,
    to_state_grp_d,
    to_state_grp_h,
    to_state_poc_d,
)

SAMPLES: int = 288  # 5 min
SATURE_H: int = 45  # minimum duration (min) of saturation to have a saturated hour
MAX_SESSION_DURATION_HOURS = 24

ID_POC: str = "id_pdc_itinerance"
ID_STATION: str = "id_station_itinerance"
ID_POOL: str = "id_pool"
SATURATION_RATIO = 0.1
OVERLOAD_RATIO = 0.2
MIN_POWER = 75

#day = date(2026,7, 5)
#day = date(2026,7, 14)
day = date(2026,8, 1)
date_file = f"{day.year}{day.month:02d}{day.day:02d}"

data_quali = "../data/"

In [2]:
def read_statics(day: date, min_power: float) -> pd.DataFrame:
    """Read static data for pocs and stations."""
    date_statics = f"{day.day:02d}-{day.month:02d}-{day.year}"
    e5_str = pd.read_csv(f"../data_DMR_e2_e3/e5_{date_statics}.csv")["extras"][0]
    statics = pd.DataFrame(json.loads(e5_str))
    statics["unite"] = statics["id_pdc_itinerance"].str[:5]
    return statics[statics["puissance_nominale"] >= min_power]

def read_statics_pools(day:date, min_power: float) -> pd.DataFrame:
    """Read static data for pocs and pools."""
    e5_statics = read_statics(day, min_power)
    return e5_statics.rename(columns={ID_STATION: ID_POOL})

In [29]:
def get_chunked_state_poc(statics, day, samples_per_day, chunk_size, sessions, statuses):
    chunks = [
        statics.iloc[i : i + chunk_size] for i in range(0, len(statics), chunk_size)
    ]
    futures = [
        get_sampled_state_poc_for_chunk(#).submit(
            day,
            samples_per_day,
            chunk,
            sessions,
            statuses,
        )  # type: ignore[call-overload]
        for chunk in chunks
    ]
    #wait(futures)

    sampled_state_poc = pd.concat(
        [future[0] for future in futures], ignore_index=True
    )
    state_poc_d = pd.concat([future[1] for future in futures], ignore_index=True)
    return (sampled_state_poc, state_poc_d)

def get_chunked_state_grp(statics, sampled_state_poc, chunk_size, id_grp, samples_per_day, saturation_ratio, overload_ratio):

    codes, _ = pd.factorize(statics[id_grp])
    statics["chunk"] = codes // chunk_size
    chunks = statics.groupby("chunk")
    #print(len(chunks))

    futures = [
        to_state_grp_d(
            to_state_grp_h(
                to_sampled_state_grp(
                    sampled_state_poc[sampled_state_poc[ID_POC].isin(chunk[ID_POC])],#.sort_values(by=["id_pdc_itinerance", "periode"]).reset_index(drop=True),
                    chunk,
                    id_grp,
                    saturation_ratio,
                    overload_ratio
                ),  # type: ignore[call-overload]
                id_grp, 
                samples_per_day,
                SATURE_H
            ),
            id_grp
        )
        for _, chunk in chunks
    ]

    state_grp = pd.concat(
        [future for future in futures], ignore_index=True
    )
    return state_grp

## test local

In [4]:
sessions = pd.read_csv('../data_test/donnees_sessions_FRHPCPNF050462_05-07-2026.csv')[['start', 'end', 'id_pdc_itinerance']]
sessions['start'] = pd.to_datetime(sessions['start'])
sessions['end'] = pd.to_datetime(sessions['end'])
statuses = pd.DataFrame({ID_POC:[], "horodatage":[], "etat_pdc":[], "occupation_pdc":[]})
statuses['horodatage'] = pd.to_datetime(statuses['horodatage'], utc=True)
# statuses = pd.DataFrame()
statics = pd.DataFrame({ID_POC:['FRHPCENF050462001', 'FRHPCENF050462002', 'FRHPCENF050462004' ], ID_STATION:['FRHPCPNF050462']*3})

samples_per_day = 288
#sampled_state_poc = get_sampled_state_poc(day, samples_per_day, sessions, statuses)

In [5]:
#sampled_state_poc[sampled_state_poc[ID_POC] == 'FRHPCENF050462001']

In [6]:
#state_poc_d = to_state_poc_d(sampled_state_poc, samples_per_day)

In [7]:
#state_poc_d, e2_pdc

In [8]:
#sample_state_station = to_sampled_state_grp(sampled_state_poc, statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO)
#state_station_h = to_state_grp_h(sample_state_station, ID_STATION, SAMPLES, SATURE_H)
#state_station_d = to_state_grp_d(state_station_h, ID_STATION)

In [9]:
#state_station_d

In [10]:
#e3_station

## test global

In [11]:
sessions_s3 = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/sessions/production.parquet", engine="pyarrow")
statuses_s3 = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/statuses/production.parquet", engine="pyarrow")

In [12]:

samples_per_day = SAMPLES #72 #288

e2_str = pd.read_csv("../data_DMR_e2_e3/e2_e3_05-07-2026.csv")["extras"][0]
e2_pdc = pd.DataFrame(json.loads(e2_str))

e3_str = pd.read_csv("../data_DMR_e2_e3/e2_e3_05-07-2026.csv")["extras"][1]
e3_station = pd.DataFrame(json.loads(e3_str))

e5_statics = read_statics(day, MIN_POWER)

# FRHPCENF050462001
e2_pdc = e2_pdc[e2_pdc[ID_POC].str[:14] == 'FRHPCENF050462']
e3_station = e3_station[e3_station[ID_STATION] == 'FRHPCPNF050462']


In [13]:
#e5_statics[e5_statics[ID_STATION] == 'FRHPCPNF080266TOTEM']
#e5_statics

In [14]:
# samples_per_day = 288

'''min_duration = timedelta(minutes=24 * 60 / samples_per_day)
max_duration = timedelta(hours=MAX_SESSION_DURATION_HOURS)
sessions = filter_sessions_duration(sessions_s3, min_duration=min_duration, max_duration=max_duration)
sessions_poc = sessions.groupby(ID_POC).agg(
                sessions_nb=NamedAgg("energy", "count"),
                energy_cum=NamedAgg("energy", "sum")
            ).reset_index()
'''


'min_duration = timedelta(minutes=24 * 60 / samples_per_day)\nmax_duration = timedelta(hours=MAX_SESSION_DURATION_HOURS)\nsessions = filter_sessions_duration(sessions_s3, min_duration=min_duration, max_duration=max_duration)\nsessions_poc = sessions.groupby(ID_POC).agg(\n                sessions_nb=NamedAgg("energy", "count"),\n                energy_cum=NamedAgg("energy", "sum")\n            ).reset_index()\n'

In [15]:
#e5_statics[[ID_POC, ID_STATION]]
'''sessions_stations = pd.merge(e5_statics[[ID_POC, ID_STATION]], sessions_poc, on=ID_POC, how='left').fillna(0)
info_sessions_stations = sessions_stations[[ID_STATION, 'sessions_nb', 'energy_cum']].groupby(ID_STATION).sum().reset_index()
info_sessions_stations
'''

"sessions_stations = pd.merge(e5_statics[[ID_POC, ID_STATION]], sessions_poc, on=ID_POC, how='left').fillna(0)\ninfo_sessions_stations = sessions_stations[[ID_STATION, 'sessions_nb', 'energy_cum']].groupby(ID_STATION).sum().reset_index()\ninfo_sessions_stations\n"

### point de recharge

In [23]:
statuses_f, sessions_f = filter_statuses_sessions(sessions_s3, statuses_s3, e5_statics)
sampled_state_poc_g = get_sampled_state_poc(day, samples_per_day, sessions_f, statuses_f)

In [17]:
chunk_size = 200
sampled_state_poc_chunk, state_poc_d_chunk = get_chunked_state_poc(e5_statics, day, samples_per_day, chunk_size, sessions_s3, statuses_s3)

In [24]:
#sampled_sessions[sampled_sessions[ID_POC] == 'FRA79E12346905331']
#sampled_statuses[sampled_statuses[ID_POC] == 'FRA79E12346905331'][100:150]
#sampled_state_poc_g[sampled_state_poc_g['pseudo_occupe'] >0]
#sampled_state_poc_g[sampled_state_poc_g[ID_POC] == 'FRA79E12346905331'][0:100]
#sessions_s3[sessions_s3[ID_POC] == 'FRA79E12346905331']
#statuses_s3[statuses_s3[ID_POC] == 'FRA79E12346905331']
#sessions_s3[sessions_s3[ID_POC] == 'FRHPCENF050462002']
sampled_state_poc_g.sort_values(by=["id_pdc_itinerance"])

,id_pdc_itinerance,periode,state
0,FR3R3E10001456611,2026-08-01 00:00:00+00:00,libre
1,FR3R3E10001456611,2026-08-01 00:05:00+00:00,libre
2,FR3R3E10001456611,2026-08-01 00:10:00+00:00,libre
3,FR3R3E10001456611,2026-08-01 00:15:00+00:00,libre
4,FR3R3E10001456611,2026-08-01 00:20:00+00:00,libre
...,...,...,...
6758984,FRZUNEFR8801ER04,2026-08-01 23:35:00+00:00,libre
6758985,FRZUNEFR8801ER04,2026-08-01 23:40:00+00:00,libre
6758986,FRZUNEFR8801ER04,2026-08-01 23:45:00+00:00,libre
6758987,FRZUNEFR8801ER04,2026-08-01 23:50:00+00:00,libre


In [19]:
sampled_state_poc_chunk.sort_values(by=["id_pdc_itinerance"])

,id_pdc_itinerance,periode,state
1482792,FR3R3E10001456611,2026-08-01 00:00:00+00:00,libre
1482793,FR3R3E10001456611,2026-08-01 00:05:00+00:00,libre
1482794,FR3R3E10001456611,2026-08-01 00:10:00+00:00,libre
1482795,FR3R3E10001456611,2026-08-01 00:15:00+00:00,libre
1482796,FR3R3E10001456611,2026-08-01 00:20:00+00:00,libre
...,...,...,...
2869629,FRZUNEFR8801ER04,2026-08-01 23:35:00+00:00,libre
2869630,FRZUNEFR8801ER04,2026-08-01 23:40:00+00:00,libre
2869631,FRZUNEFR8801ER04,2026-08-01 23:45:00+00:00,libre
2869632,FRZUNEFR8801ER04,2026-08-01 23:50:00+00:00,libre


In [25]:
state_poc_d_g = to_state_poc_d(sampled_state_poc_g, samples_per_day)
state_poc_d_g

,id_pdc_itinerance,occupe,hors_service,libre
0,FR3R3E10001456611,70.0,0.0,1370.0
1,FR3R3E10001456612,80.0,0.0,1360.0
2,FRALDE100541,145.0,0.0,1295.0
3,FRALDE100551,170.0,0.0,1270.0
4,FRALDE100552,90.0,0.0,1350.0
...,...,...,...,...
23462,FRZUNEFR8601ER06,220.0,15.0,1205.0
23463,FRZUNEFR8801ER01,175.0,0.0,1265.0
23464,FRZUNEFR8801ER02,140.0,0.0,1300.0
23465,FRZUNEFR8801ER03,65.0,0.0,1375.0


In [26]:
state_poc_d_chunk.sort_values(by=["id_pdc_itinerance"])

,id_pdc_itinerance,occupe,hors_service,libre
5148,FR3R3E10001456611,70.0,0.0,1370.0
12750,FR3R3E10001456612,80.0,0.0,1360.0
14576,FRALDE100541,145.0,0.0,1295.0
14085,FRALDE100551,170.0,0.0,1270.0
12419,FRALDE100552,90.0,0.0,1350.0
...,...,...,...,...
8870,FRZUNEFR8601ER06,220.0,15.0,1205.0
2261,FRZUNEFR8801ER01,175.0,0.0,1265.0
14413,FRZUNEFR8801ER02,140.0,0.0,1300.0
5938,FRZUNEFR8801ER03,65.0,0.0,1375.0


### station

In [28]:
sampled_state_station_g = to_sampled_state_grp(sampled_state_poc_g, e5_statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO)
print(len(sampled_state_station_g))

1581120


In [ ]:
#sampled_state_station_g.sort_values(by=[ID_STATION, "periode"])[100:150]

In [30]:
chunk_size = 200
state_station_chunk = get_chunked_state_grp(e5_statics, sampled_state_poc_g, chunk_size, ID_STATION, SAMPLES, SATURATION_RATIO, OVERLOAD_RATIO)


In [31]:
state_station_chunk.sort_values(by=[ID_STATION])

,id_station_itinerance,periode,nb_pdc,nb_h,hs,inactif,sature_cum,sature_max,surcharge,actif
2871,FR3R3P89882136,2026-08-01,2,24,0.0,1290.0,0.0,0.0,0.0,150.0
1994,FRALDPFR00916,2026-08-01,8,24,0.0,1220.0,0.0,0.0,0.0,220.0
3221,FRALDPFR00950,2026-08-01,8,24,0.0,995.0,0.0,0.0,5.0,440.0
919,FRALLPGO000007,2026-08-01,6,24,0.0,830.0,205.0,55.0,170.0,235.0
3925,FRALLPGO000013,2026-08-01,10,24,0.0,395.0,0.0,0.0,40.0,1005.0
...,...,...,...,...,...,...,...,...,...,...
3045,FRZUNP6023950095875504781,2026-08-01,8,24,0.0,570.0,25.0,10.0,60.0,785.0
1822,FRZUNP6927750076048540479,2026-08-01,8,24,0.0,1090.0,0.0,0.0,0.0,350.0
367,FRZUNP7329346578064027187,2026-08-01,25,24,0.0,175.0,0.0,0.0,0.0,1265.0
368,FRZUNP8610050047391683219,2026-08-01,6,24,0.0,1000.0,20.0,10.0,115.0,305.0


In [32]:
state_station_h = to_state_grp_h(sampled_state_station_g, ID_STATION, SAMPLES, SATURE_H)
state_station_d = to_state_grp_d(state_station_h, ID_STATION)

state_station_d

,id_station_itinerance,periode,nb_pdc,nb_h,hs,inactif,sature_cum,sature_max,surcharge,actif
0,FR3R3P89882136,2026-08-01,2,24,0.0,1290.0,0.0,0.0,0.0,150.0
1,FRALDPFR00916,2026-08-01,8,24,0.0,1220.0,0.0,0.0,0.0,220.0
2,FRALDPFR00950,2026-08-01,8,24,0.0,995.0,0.0,0.0,5.0,440.0
3,FRALLPGO000007,2026-08-01,6,24,0.0,830.0,205.0,55.0,170.0,235.0
4,FRALLPGO000013,2026-08-01,10,24,0.0,395.0,0.0,0.0,40.0,1005.0
...,...,...,...,...,...,...,...,...,...,...
5485,FRZUNP6023950095875504781,2026-08-01,8,24,0.0,570.0,25.0,10.0,60.0,785.0
5486,FRZUNP6927750076048540479,2026-08-01,8,24,0.0,1090.0,0.0,0.0,0.0,350.0
5487,FRZUNP7329346578064027187,2026-08-01,25,24,0.0,175.0,0.0,0.0,0.0,1265.0
5488,FRZUNP8610050047391683219,2026-08-01,6,24,0.0,1000.0,20.0,10.0,115.0,305.0
